# CNN-Transformer Smart Insole — Gait Estimation
## Leave-One-Subject-Out (LOSO) Cross-Validation

Architecture: **CNN spatial encoder → Positional Encoding → Transformer Encoder → mean pooling → FC**

This notebook replaces the LSTM+Attention block from `part2-cnn_lstm_loso.ipynb` with a Transformer Encoder.
All data loading, preprocessing, training loop, and evaluation code are identical to that notebook.

> **Note:** Subjects 1 and 3 were excluded due to battery-induced timestamp drift. LOSO runs on the three remaining subjects.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import random
import math
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

BASE       = Path("data")
SYNCED_DIR = BASE / "synced"
MODELS_DIR = BASE / "models_transformer_v3"   # v3: velocity diff channels + SEQ_LEN=60
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
print("Synced dir:", SYNCED_DIR)
print("Models dir:", MODELS_DIR)

## 2. Joint setup and skeleton connections

In [ ]:
TARGET_JOINTS = [
    "left-hip",
    "right-hip",
    "left-knee",
    "right-knee",
    "left-heel",
    "right-heel",
    "left-big-toe-tip",
    "right-big-toe-tip",
]
OUTPUT_JOINTS = TARGET_JOINTS.copy()

CONNECTIONS = [
    ("left-hip",  "left-knee"),
    ("left-knee", "left-heel"),
    ("left-heel", "left-big-toe-tip"),
    ("right-hip",  "right-knee"),
    ("right-knee", "right-heel"),
    ("right-heel", "right-big-toe-tip"),
    ("left-hip",  "right-hip"),
]

def as_str_list(names):
    # NumPy saves strings as bytes (e.g. b'left-hip') when loading .npz with allow_pickle=True.
    # This helper normalises every element: bytes → decoded UTF-8 str, anything else → str().
    # Returns: a plain Python list of strings, e.g. ['left-hip', 'right-hip', 'left-knee', ...]
    # Without this, joint-name comparisons like `name in mapping` would silently fail
    # because b'left-hip' != 'left-hip'.
    return [n.decode("utf-8") if isinstance(n, bytes) else str(n) for n in names]

## 3. Load synced sessions

In [ ]:
def load_synced_sessions(synced_dir: Path):
    session_files = sorted(synced_dir.glob("*_synced.npz"))
    if not session_files:
        raise FileNotFoundError(f"No synced files found in {synced_dir}")
    sessions = []
    for path in session_files:
        data = np.load(path, allow_pickle=True)
        insoles     = np.asarray(data["insoles"],     dtype=np.float32)
        joints      = np.asarray(data["joints"],      dtype=np.float32)
        joint_names = as_str_list(data["joint_names"])
        timestamps  = np.asarray(data["timestamps"],  dtype=np.float64)
        sessions.append({
            "name":        path.stem.replace("_synced", ""),
            "insoles":     insoles,
            "joints":      joints,
            "joint_names": joint_names,
            "timestamps":  timestamps,
        })
        print(f"{path.name}: insoles={insoles.shape}, joints={joints.shape}")
    return sessions

sessions = load_synced_sessions(SYNCED_DIR)
print(f"\nTotal subjects loaded: {len(sessions)}")
assert len(sessions) >= 2, (
    f"Need at least 2 subjects for LOSO, found {len(sessions)}."
)

## 4. Session preprocessing

In [ ]:
def replace_nan_with_zero(x):
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def build_joint_index(joint_names):
    mapping = {name: i for i, name in enumerate(joint_names)}
    missing = [name for name in TARGET_JOINTS if name not in mapping]
    if missing:
        raise ValueError(f"Missing required joints: {missing}")
    return mapping

def select_output_targets(joints, joint_names):
      # SAM 3D Body outputs 3D coordinates for the FULL body (many joints).
    # We only care about 8 lower-limb joints defined in OUTPUT_JOINTS.
    # This function:
    #   1. Uses build_joint_index to find which columns correspond to our 8 joints.
    #   2. Slices the joints array to keep only those 8, in the fixed ORDER of OUTPUT_JOINTS.
    #      (fixed order is critical: it must match the order the model's FC head outputs)
    # Input  shape: (T, all_joints, 3)  — T frames(time), many joints, xyz coords
    # Output shape: (T, 8, 3)           — T frames, 8 selected joints, xyz coords
    name_to_idx  = build_joint_index(joint_names)
    keep_indices = [name_to_idx[name] for name in OUTPUT_JOINTS]
    return joints[:, keep_indices, :].astype(np.float32)

def preprocess_sessions(sessions):
      # Applies the above cleaning and selection steps to every loaded session,
    # then packages each into a standardised dict with these keys:
    #
    #   "name"       — subject name string (e.g. "brahim")
    #   "X"          — insole pressure maps,  shape (T, 2, 33, 15), float32
    #                  channel 0 = left insole (33×15 pressure map)
    #                  channel 1 = right insole (33×15 pressure map)
    #   "Y"          — selected joint 3D coords, shape (T, 8, 3), float32
    #                  NOT yet normalised here; per-fold z-score normalisation
    #                  is applied later inside the LOSO loop (Section 9)
    #   "Y_raw"      — identical copy of Y before any normalisation.
    #                  kept separately so evaluation metrics can be computed
    #                  in original (unnormalised) coordinate space
    #   "timestamps" — frame timestamps in seconds, shape (T,), float64
    #                  used for alignment verification, not fed to the model
    processed = []
    for session in sessions:
        x = replace_nan_with_zero(session["insoles"])
        y = select_output_targets(session["joints"], session["joint_names"])
        processed.append({
            "name":       session["name"],
            "X":          x,
            "Y":          y,
            "Y_raw":      y.copy(),
            "timestamps": session["timestamps"],
        })
        print(f"{session['name']}: X={x.shape}, Y={y.shape}")
    return processed

processed_sessions = preprocess_sessions(sessions)

## 5. Model configuration

Transformer-specific hyperparameters replace the LSTM ones.

In [ ]:
SEQ_LEN     = 60
# [v3 change] Increased from 30 → 60 frames (~2 seconds at 30 fps).
# A human gait cycle (one full heel-strike to heel-strike) takes ~1.0–1.5 s.
# SEQ_LEN=30 (~1 s) often captured less than one full cycle.
# SEQ_LEN=60 (~2 s) guarantees the model always sees at least one complete
# stride, giving it enough context to distinguish gait phases reliably.
# Trade-off: each training sample is 2× longer → slightly fewer samples
# and slightly more GPU memory per batch, but both are negligible here.

SEQ_STRIDE  = 1
BATCH_SIZE  = 32
EPOCHS      = 400
PATIENCE    = 20
VAL_RATIO   = 0.15

# CNN encoder
CNN_CHANNELS = (16, 32)

# Transformer
D_MODEL                = 128
NHEAD                  = 4
NUM_TRANSFORMER_LAYERS = 2
DIM_FEEDFORWARD        = 256
DROPOUT                = 0.2

print("Config OK")
print(f"  SEQ_LEN={SEQ_LEN}  ← increased from 30 (v2) to 60 (v3)")
print(f"  BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, PATIENCE={PATIENCE}")
print(f"  D_MODEL={D_MODEL}, NHEAD={NHEAD}, LAYERS={NUM_TRANSFORMER_LAYERS}, FFN={DIM_FEEDFORWARD}")
print(f"  DROPOUT={DROPOUT}")

## 6. Build temporal sequences

In [ ]:
class SlidingWindowDataset(Dataset):
    def __init__(self, sessions, items, seq_len):
        self.sessions = sessions
        self.items    = items
        self.seq_len  = seq_len

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        session_idx, end_idx = self.items[idx]
        s         = self.sessions[session_idx]
        start_idx = end_idx - self.seq_len + 1

        # Raw pressure window — shape (T, 2, H, W)
        x = s["X"][start_idx : end_idx + 1]

        # ── [v3] Velocity (frame-to-frame difference) channels ──────────────
        # x_diff[t] = x[t] - x[t-1]  →  how fast pressure is changing
        #
        # Why this helps:
        #   The error heatmap showed spikes at gait phase transitions
        #   (heel-strike, toe-off). At these moments the pressure map
        #   CHANGES rapidly, but the raw values alone don't encode that speed.
        #   Adding the difference gives the model an explicit "velocity" signal:
        #     positive diff  →  pressure increasing  (foot landing)
        #     negative diff  →  pressure decreasing  (foot lifting)
        #   This lets the Transformer detect transition events directly,
        #   rather than having to infer them from two consecutive raw frames.
        #
        # prepend=x[:1] repeats the first frame so that x_diff[0] = 0
        # (no previous frame to compare against), keeping shape (T, 2, H, W).
        x_diff = np.diff(x, axis=0, prepend=x[:1])   # (T, 2, H, W)

        # Concatenate raw + diff along the channel axis → (T, 4, H, W)
        #   channel 0: left insole  pressure
        #   channel 1: right insole pressure
        #   channel 2: left insole  pressure change  (velocity)
        #   channel 3: right insole pressure change  (velocity)
        x_input = np.concatenate([x, x_diff], axis=1).astype(np.float32)

        y     = s["Y"][end_idx]
        y_raw = s["Y_raw"][end_idx]
        return {
            "X":       torch.from_numpy(x_input).float(),
            "Y":       torch.from_numpy(y).float(),
            "Y_raw":   torch.from_numpy(y_raw).float(),
            "session": s["name"],
        }


def build_items(sessions_subset, seq_len):
    items = []
    for local_i, s in enumerate(sessions_subset):
        T = len(s["X"])
        if T < seq_len:
            print(f"  WARNING: {s['name']} has only {T} frames < SEQ_LEN={seq_len}, skipped.")
            continue
        for end_idx in range(seq_len - 1, T):
            items.append((local_i, end_idx))
    return items

print("Dataset helpers defined.")
print("Input channels per frame: 4  (2 pressure + 2 velocity diff)")

## 7. CNN-Transformer model

```
Input  (B, T, 4, H, W)   ← [v3] 4 channels: 2 pressure + 2 velocity diff
  │
  ├─ CNN FrameEncoder  ──► (B, T, 32)   [spatial features per frame]
  │
  ├─ Linear projection ──► (B, T, D_MODEL=128)
  │
  ├─ Sinusoidal Positional Encoding
  │
  ├─ Transformer Encoder (N layers, nhead=4)
  │       └─ Self-attention across T=60 timesteps (~2 seconds)
  │
  ├─ Mean pooling over T ──► (B, D_MODEL)
  │
  └─ FC ──► (B, 8×3 = 24)
```

In [ ]:
# ── Sinusoidal positional encoding ──────────────────────────
class PositionalEncoding(nn.Module):
    """
    Standard sine/cosine positional encoding (Vaswani et al., 2017).
    Adds a fixed position signal to each token so the Transformer
    knows which timestep each frame belongs to.
    """
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))   # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


# ── CNN spatial encoder ──────────────────────────────────────
class FrameEncoder(nn.Module):
    """
    Extracts a spatial feature vector from one insole pressure frame.
    Input : (B, in_channels, H, W)
    Output: (B, c2)
    [v3] in_channels is now 4 (2 pressure + 2 velocity diff).
    """
    def __init__(self, in_channels: int = 4, c1: int = 16, c2: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, c1, kernel_size=3, padding=1),
            nn.BatchNorm2d(c1), nn.ReLU(),
            nn.Conv2d(c1, c1, kernel_size=3, padding=1),
            nn.BatchNorm2d(c1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(c1, c2, kernel_size=3, padding=1),
            nn.BatchNorm2d(c2), nn.ReLU(),
            nn.Conv2d(c2, c2, kernel_size=3, padding=1),
            nn.BatchNorm2d(c2), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.flatten(self.encoder(x), start_dim=1)   # (B, c2)


# ── CNN-Transformer regressor ────────────────────────────────
class CNNTransformerRegressor(nn.Module):
    def __init__(
        self,
        output_dim: int,
        in_channels: int = 4,    # [v3] 4 = 2 pressure + 2 velocity diff
        c1: int = 16,
        c2: int = 32,
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 2,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
    ):
        super().__init__()

        # 1. Spatial encoder — accepts in_channels input channels
        self.cnn = FrameEncoder(in_channels=in_channels, c1=c1, c2=c2)
        with torch.no_grad():
            dummy       = torch.zeros(1, in_channels, 66, 30)
            encoder_dim = self.cnn(dummy).shape[1]
        print(f"CNN encoder_dim = {encoder_dim}  (in_channels={in_channels})")

        # 2. Project CNN output to Transformer token dimension
        self.input_proj = nn.Linear(encoder_dim, d_model)

        # 3. Positional encoding
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)

        # 4. Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = d_model,
            nhead           = nhead,
            dim_feedforward = dim_feedforward,
            dropout         = dropout,
            batch_first     = True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm        = nn.LayerNorm(d_model)

        # 5. Output head
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(d_model, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        x = self.cnn(x)             # (B*T, encoder_dim)
        x = x.view(B, T, -1)       # (B, T, encoder_dim)
        x = self.input_proj(x)     # (B, T, d_model)
        x = self.pos_enc(x)        # (B, T, d_model)
        x = self.transformer(x)    # (B, T, d_model)
        x = self.norm(x)
        x = x.mean(dim=1)          # (B, d_model)
        x = self.dropout(x)
        return self.fc(x)          # (B, output_dim)


# ── Factory helpers ──────────────────────────────────────────
def make_model():
    output_dim = len(OUTPUT_JOINTS) * 3
    m = CNNTransformerRegressor(
        output_dim      = output_dim,
        in_channels     = 4,             # [v3] pressure + velocity diff
        c1              = CNN_CHANNELS[0],
        c2              = CNN_CHANNELS[1],
        d_model         = D_MODEL,
        nhead           = NHEAD,
        num_layers      = NUM_TRANSFORMER_LAYERS,
        dim_feedforward = DIM_FEEDFORWARD,
        dropout         = DROPOUT,
    ).to(DEVICE)
    return m


def make_optimizer(model):
    return torch.optim.Adam([
        {"params": model.cnn.parameters(),         "lr": 2e-3},
        {"params": model.input_proj.parameters(),  "lr": 1e-3},
        {"params": model.transformer.parameters(), "lr": 5e-4},
        {"params": model.fc.parameters(),          "lr": 1e-3},
    ], weight_decay=1e-4)


# Sanity check
_m = make_model()
n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
del _m

## 8. Loss function and training helpers

In [ ]:
JOINT_WEIGHTS = torch.tensor([
    0.5, 0.5,   # hips  — already easy to predict
    1.0, 1.0,   # knees
    2.0, 2.0,   # heels
    2.0, 2.0,   # toes
], dtype=torch.float32)


def weighted_smoothl1(pred, target):
    pred_j   = pred.view(-1, 8, 3)
    target_j = target.view(-1, 8, 3)
    per_joint = torch.abs(pred_j - target_j).mean(dim=2)   # (B, 8)
    w = JOINT_WEIGHTS.to(pred.device)
    return (per_joint * w).mean()


criterion = weighted_smoothl1


def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, n_samples = 0.0, 0
    with torch.set_grad_enabled(is_train):
        for batch in loader:
            x    = batch["X"].to(DEVICE)
            y    = batch["Y"].to(DEVICE).view(batch["X"].size(0), -1)
            pred = model(x)
            loss = criterion(pred, y)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            bs = x.size(0)
            total_loss += loss.item() * bs
            n_samples  += bs
    return total_loss / max(n_samples, 1)


def evaluate(model, loader, y_mean=None, y_std=None):
    model.eval()
    total_loss, n_samples = 0.0, 0
    preds_list, trues_list = [], []
    with torch.no_grad():
        for batch in loader:
            x     = batch["X"].to(DEVICE)
            y     = batch["Y"].to(DEVICE).view(x.size(0), -1)
            y_raw = batch["Y_raw"].cpu().numpy()
            pred  = model(x)
            loss  = criterion(pred, y)
            bs    = x.size(0)
            total_loss += loss.item() * bs
            n_samples  += bs
            pred_raw = pred.view(bs, len(OUTPUT_JOINTS), 3).cpu().numpy()
            if y_mean is not None and y_std is not None:
                pred_raw = pred_raw * y_std + y_mean
            preds_list.append(pred_raw)
            trues_list.append(y_raw)

    preds = np.concatenate(preds_list, axis=0)
    trues = np.concatenate(trues_list, axis=0)
    mse   = float(np.mean((preds - trues) ** 2))
    mae   = float(np.mean(np.abs(preds - trues)))
    per_joint_mae = np.mean(np.abs(preds - trues), axis=(0, 2))   # (J,)
    return {
        "loss":          total_loss / max(n_samples, 1),
        "mse":           mse,
        "mae":           mae,
        "per_joint_mae": per_joint_mae,
        "preds":         preds,
        "trues":         trues,
    }

print("Helpers defined.")

## 9. LOSO Cross-Validation

**Protocol:**
- Fold *k*: subject *k* → test set
- Remaining subjects → training pool
- `VAL_RATIO` of training sequences held out for early stopping
- Model re-initialised from scratch each fold

In [ ]:
n_subjects     = len(processed_sessions)
fold_results   = []
fold_histories = []

print("=" * 60)
print("CNN-Transformer v3 LOSO Cross-Validation")
print("Changes vs v2: in_channels 2→4 (velocity diff) | SEQ_LEN 30→60")
print("=" * 60)

for test_subj_idx in range(n_subjects):

    test_name   = processed_sessions[test_subj_idx]["name"]
    train_idxs  = [i for i in range(n_subjects) if i != test_subj_idx]
    train_names = [processed_sessions[i]["name"] for i in train_idxs]

    print(f"\n{'─' * 60}")
    print(f"Fold {test_subj_idx + 1}/{n_subjects}  |  "
          f"Test: [{test_name}]  |  Train: {train_names}")
    print(f"{'─' * 60}")

    # ── Session subsets ──────────────────────────────────────
    train_sessions = [copy.deepcopy(processed_sessions[i]) for i in train_idxs]
    test_sessions  = [copy.deepcopy(processed_sessions[test_subj_idx])]

    # Normalise Y using train statistics only
    train_Y = np.concatenate([s["Y"] for s in train_sessions], axis=0)
    Y_mean  = train_Y.mean(axis=0)
    Y_std   = train_Y.std(axis=0) + 1e-6
    for s in train_sessions:
        s["Y"] = (s["Y"] - Y_mean) / Y_std
    for s in test_sessions:
        s["Y"] = (s["Y"] - Y_mean) / Y_std

    # ── Sliding-window items ─────────────────────────────────
    all_train_items = build_items(train_sessions, SEQ_LEN)
    test_items      = build_items(test_sessions,  SEQ_LEN)

    random.seed(SEED)
    random.shuffle(all_train_items)
    n_val            = max(1, int(len(all_train_items) * VAL_RATIO))
    val_items_fold   = all_train_items[-n_val:]
    train_items_fold = all_train_items[:-n_val]

    print(f"  Train sequences : {len(train_items_fold)}")
    print(f"  Val   sequences : {len(val_items_fold)}")
    print(f"  Test  sequences : {len(test_items)}")

    # ── DataLoaders ──────────────────────────────────────────
    kw = dict(batch_size=BATCH_SIZE, num_workers=0,
              pin_memory=torch.cuda.is_available())
    train_loader = DataLoader(
        SlidingWindowDataset(train_sessions, train_items_fold, SEQ_LEN),
        shuffle=True, **kw)
    val_loader = DataLoader(
        SlidingWindowDataset(train_sessions, val_items_fold, SEQ_LEN),
        shuffle=False, **kw)
    test_loader = DataLoader(
        SlidingWindowDataset(test_sessions, test_items, SEQ_LEN),
        shuffle=False, **kw)

    # ── Fresh model, optimiser, and scheduler ────────────────
    torch.manual_seed(SEED)
    model     = make_model()
    optimizer = make_optimizer(model)

    # [v2 change] CosineAnnealingLR: smoothly decays the learning rate
    # from its initial value down to eta_min over T_max epochs.
    # Cosine shape means LR drops slowly at first, faster in the middle,
    # then slowly again at the end — allowing fine-grained convergence
    # without the val loss oscillations seen in v1 (fixed LR).
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-5
    )

    model_path = MODELS_DIR / f"cnn_transformer_v3_fold{test_subj_idx+1}_{test_name}.pt"

    # ── Training loop with early stopping ───────────────────
    history       = {"train_loss": [], "val_loss": [], "lr": []}
    best_val      = float("inf")
    best_epoch    = -1
    patience_left = PATIENCE

    for epoch in range(1, EPOCHS + 1):
        train_loss = run_epoch(model, train_loader, optimizer=optimizer)
        val_loss   = run_epoch(model, val_loader,   optimizer=None)
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["lr"].append(current_lr)

        # Print every epoch; also show LR at epoch 1 and every 50 epochs
        show_lr = (epoch == 1 or epoch % 50 == 0)
        print(f"  Epoch {epoch:03d}/{EPOCHS} | "
              f"train={train_loss:.6f} | val={val_loss:.6f}"
              + (f" | lr={current_lr:.2e}" if show_lr else ""), end="")

        if val_loss < best_val:
            best_val      = val_loss
            best_epoch    = epoch
            patience_left = PATIENCE
            torch.save(model.state_dict(), model_path)
            print("  ✓", end="")
        else:
            patience_left -= 1
        print()

        # [v2] Step the scheduler every epoch regardless of early stopping.
        # This ensures the LR cosine curve is consistent even when training
        # terminates early — the saved checkpoint always used the scheduled LR.
        scheduler.step()

        if patience_left <= 0:
            print(f"  Early stopping at epoch {epoch}.")
            break

    print(f"  Best epoch: {best_epoch},  best val loss: {best_val:.6f}")

    # ── Evaluate best checkpoint on test set ─────────────────
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    results = evaluate(model, test_loader, y_mean=Y_mean, y_std=Y_std)

    print(f"\n  ── Fold {test_subj_idx+1} Test Results ──")
    print(f"  Test subject : {test_name}")
    print(f"  Test loss    : {results['loss']:.6f}")
    print(f"  Test MSE     : {results['mse']:.6f}")
    print(f"  Test MAE     : {results['mae']:.6f}")
    print(f"  Per-joint MAE:")
    for jname, jmae in zip(OUTPUT_JOINTS, results["per_joint_mae"]):
        print(f"    {jname:<22s}: {jmae:.4f}")

    fold_results.append({
        "fold":       test_subj_idx + 1,
        "test_name":  test_name,
        "best_epoch": best_epoch,
        "best_val":   best_val,
        **{k: results[k] for k in ("loss", "mse", "mae", "per_joint_mae", "preds", "trues")},
    })
    fold_histories.append(history)

## 10. Aggregated results

In [ ]:
mses    = [r["mse"] for r in fold_results]
maes    = [r["mae"] for r in fold_results]
pj_maes = np.stack([r["per_joint_mae"] for r in fold_results])   # (n_folds, J)

print("=" * 60)
print("CNN-Transformer LOSO — Aggregated Results")
print("=" * 60)

print(f"\n{'Fold':<6} {'Subject':<16} {'MSE':>10} {'MAE':>10}")
print("─" * 46)
for r in fold_results:
    print(f"  {r['fold']:<4} {r['test_name']:<16} "
          f"{r['mse']:>10.6f} {r['mae']:>10.6f}")
print("─" * 46)
print(f"  {'Mean':<20} {np.mean(mses):>10.6f} {np.mean(maes):>10.6f}")
print(f"  {'Std':<20} {np.std(mses):>10.6f}  {np.std(maes):>10.6f}")

print("\nPer-joint MAE — mean ± std across folds:")
for j, jname in enumerate(OUTPUT_JOINTS):
    vals = pj_maes[:, j]
    print(f"  {jname:<22s}: {vals.mean():.4f} ± {vals.std():.4f}")

## 11. Loss curves

In [ ]:
fig, axes = plt.subplots(1, len(fold_histories), figsize=(5 * len(fold_histories), 4))
if len(fold_histories) == 1:
    axes = [axes]

for ax, history, r in zip(axes, fold_histories, fold_results):
    ep = range(1, len(history["train_loss"]) + 1)
    ax.plot(ep, history["train_loss"], label="train")
    ax.plot(ep, history["val_loss"],   label="val")
    ax.axvline(r["best_epoch"], color="red", linestyle="--", linewidth=0.8, label=f"best ep {r['best_epoch']}")
    ax.set_title(f"Fold {r['fold']} — test: {r['test_name']}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("CNN-Transformer — Training Curves", fontweight="bold")
plt.tight_layout()
plt.savefig("transformer_loss_curves.png", dpi=150)
plt.show()

## 12. Per-joint MAE bar chart

In [ ]:
mean_pj = pj_maes.mean(axis=0)
std_pj  = pj_maes.std(axis=0)
short_names = [j.replace("left-","L-").replace("right-","R-")
                .replace("-big-toe-tip","Toe") for j in OUTPUT_JOINTS]

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(OUTPUT_JOINTS))
ax.bar(x, mean_pj, yerr=std_pj, capsize=4, color="steelblue", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(short_names, rotation=30, ha="right")
ax.set_ylabel("MAE (normalised coords)")
ax.set_title("CNN-Transformer — Per-joint MAE (mean ± std over folds)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("transformer_per_joint_mae.png", dpi=150)
plt.show()

## 13. Skeleton visualisation and gait animation

In [ ]:
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D

CONN_IDX = [
    (0, 2), (2, 4), (4, 6),
    (1, 3), (3, 5), (5, 7),
    (0, 1),
]


def plot_skeleton_comparison(preds, trues, frame_idx=0, fold_name="subject"):
    fig = plt.figure(figsize=(14, 5))
    titles     = ["Ground Truth", "Prediction", "Overlay (GT=blue, Pred=red)"]
    data_pairs = [
        (trues[frame_idx], None),
        (preds[frame_idx], None),
        (trues[frame_idx], preds[frame_idx]),
    ]
    for col, (title, (d1, d2)) in enumerate(zip(titles, data_pairs)):
        ax = fig.add_subplot(1, 3, col + 1, projection="3d")

        def draw(pts, color, alpha=1.0):
            ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=color, s=50, alpha=alpha)
            for i, j in CONN_IDX:
                ax.plot([pts[i,0], pts[j,0]], [pts[i,1], pts[j,1]],
                        [pts[i,2], pts[j,2]], c=color, lw=2, alpha=alpha)

        draw(d1, "#2196F3")
        if d2 is not None:
            draw(d2, "#F44336", alpha=0.7)
        ax.set_title(f"{title}\n[{fold_name}] frame {frame_idx}", fontsize=9)
        ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
        ax.view_init(elev=15, azim=-70)
    plt.tight_layout()
    plt.savefig(f"transformer_skeleton_{fold_name}.png", dpi=150)
    plt.show()


def plot_joint_trajectories(preds, trues, fold_name="subject", n_frames=300):
    preds, trues = preds[:n_frames], trues[:n_frames]
    t = np.arange(len(preds))
    fig, axes = plt.subplots(8, 3, figsize=(15, 20))
    fig.suptitle(f"Joint Trajectories — [{fold_name}]", fontsize=12, fontweight="bold")
    for j, jname in enumerate(OUTPUT_JOINTS):
        short = jname.replace("left-","L-").replace("right-","R-")
        for ax_i, ax_n in enumerate(["X", "Y", "Z"]):
            ax = axes[j, ax_i]
            ax.plot(t, trues[:, j, ax_i], "b-",  lw=1.5, label="GT")
            ax.plot(t, preds[:, j, ax_i], "r--", lw=1.2, label="Pred")
            if ax_i == 0:
                ax.set_ylabel(f"{short}\n{ax_n}", fontsize=7)
            if j == 0 and ax_i == 0:
                ax.legend(fontsize=6)
            ax.tick_params(labelsize=6)
            ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"transformer_trajectories_{fold_name}.png", dpi=150)
    plt.show()


def plot_error_heatmap(preds, trues, fold_name="subject"):
    per_frame_mae = np.mean(np.abs(preds - trues), axis=2)   # (N, 8)
    fig, ax = plt.subplots(figsize=(16, 4))
    im = ax.imshow(per_frame_mae.T, aspect="auto", cmap="RdYlGn_r")
    ax.set_yticks(range(8))
    ax.set_yticklabels(
        [j.replace("left-","L-").replace("right-","R-").replace("-big-toe-tip","Toe")
         for j in OUTPUT_JOINTS], fontsize=9)
    ax.set_xlabel("Frame")
    ax.set_title(f"Per-frame Per-joint MAE — [{fold_name}]")
    plt.colorbar(im, ax=ax, label="MAE")
    plt.tight_layout()
    plt.savefig(f"transformer_heatmap_{fold_name}.png", dpi=150)
    plt.show()


def animate_gait(preds, trues, fold_name="subject", n_frames=120, interval=80):
    preds, trues = preds[:n_frames], trues[:n_frames]
    all_pts = np.concatenate([preds, trues], axis=0)
    lims    = [(all_pts[:,:,i].min()-0.1, all_pts[:,:,i].max()+0.1) for i in range(3)]
    fig = plt.figure(figsize=(12, 5))
    ax1, ax2 = fig.add_subplot(121, projection="3d"), fig.add_subplot(122, projection="3d")
    for ax, title in [(ax1, f"GT [{fold_name}]"), (ax2, f"Pred [{fold_name}]")]:
        ax.set_xlim(*lims[0]); ax.set_ylim(*lims[1]); ax.set_zlim(*lims[2])
        ax.set_title(title, fontsize=10)
        ax.view_init(elev=15, azim=-70)
    scat1 = ax1.scatter([], [], [], c="#2196F3", s=50)
    scat2 = ax2.scatter([], [], [], c="#F44336", s=50)
    lines1 = [ax1.plot([], [], [], "b-", lw=2)[0] for _ in CONN_IDX]
    lines2 = [ax2.plot([], [], [], "r-", lw=2)[0] for _ in CONN_IDX]
    txt    = fig.text(0.5, 0.02, "", ha="center", fontsize=9)

    def update(frame):
        for pts, scat, lines in [(trues[frame], scat1, lines1), (preds[frame], scat2, lines2)]:
            scat._offsets3d = (pts[:,0], pts[:,1], pts[:,2])
            for (i, j), line in zip(CONN_IDX, lines):
                line.set_data([pts[i,0], pts[j,0]], [pts[i,1], pts[j,1]])
                line.set_3d_properties([pts[i,2], pts[j,2]])
        txt.set_text(f"Frame {frame}/{n_frames}")
        return [scat1, scat2] + lines1 + lines2 + [txt]

    ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=interval, blit=True)
    ani.save(f"transformer_gait_{fold_name}.gif", writer="pillow", fps=1000 // interval)
    print(f"Saved: transformer_gait_{fold_name}.gif")
    plt.show()
    return ani


# Run visualisations for every fold
for result in fold_results:
    name  = result["test_name"]
    preds = result["preds"]
    trues = result["trues"]
    print(f"\n=== Visualising [{name}] ===")
    plot_skeleton_comparison(preds, trues, frame_idx=50, fold_name=name)
    plot_joint_trajectories(preds, trues, fold_name=name, n_frames=300)
    plot_error_heatmap(preds, trues, fold_name=name)
    animate_gait(preds, trues, fold_name=name, n_frames=120)

## 14. LaTeX summary table

In [ ]:
print(r"\begin{table}[h]")
print(r"\centering")
print(r"\begin{tabular}{lcc}")
print(r"\toprule")
print(r"Test Subject & MSE & MAE \\")
print(r"\midrule")
for r in fold_results:
    print(f"  {r['test_name']} (Fold {r['fold']}) & "
          f"{r['mse']:.4f} & {r['mae']:.4f} \\\\")
print(r"\midrule")
print(f"  Mean $\\pm$ Std & "
      f"{np.mean(mses):.4f} $\\pm$ {np.std(mses):.4f} & "
      f"{np.mean(maes):.4f} $\\pm$ {np.std(maes):.4f} \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\caption{CNN-Transformer LOSO results. MSE and MAE on normalised joint coordinates.}")
print(r"\label{tab:transformer_loso}")
print(r"\end{table}")